# Install mHC package from GitHub
!pip install -q git+https://github.com/ashishjv1/mHC.git
!pip install -q wandb

---
## 0. Setup

In [ ]:
# Install mHC package from GitHub
!pip install -q git+https://github.com/ashishpatel26/mHC.git
!pip install -q wandb

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import wandb

matplotlib.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

from src.newton_schulz import newton_schulz_muon, newton_schulz_polar, retract_to_stiefel
from src.muon import Muon
from src.model import GPT
from configs.train_config import TrainConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# W&B login
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass  # Not on Colab, or key not set — wandb.login() will prompt

wandb.login()

---
## Part 1: The Newton-Schulz Trick — SVD Without the SVD

### The core idea

Given a matrix $G$, the **polar decomposition** is $G = UP$ where $U$ is orthogonal and $P$ is symmetric positive semi-definite. If $G = V\Sigma W^T$ is the SVD, then $U = VW^T$ — the polar factor sets all singular values to 1.

Computing the SVD is $O(n^3)$ and not GPU-friendly. Instead, Newton-Schulz applies a **matrix polynomial** iteratively:

$$X_{k+1} = X_k(aI + bX_k^TX_k + c(X_k^TX_k)^2)$$

In terms of singular values, each iteration applies the scalar polynomial:

$$\sigma \mapsto \sigma(a + b\sigma^2 + c\sigma^4)$$

The magic is in choosing $(a, b, c)$ so that $\sigma = 1$ is the **stable fixed point**.

### 1.1 The Quintic Polynomial

Let's visualize $p(\sigma) = \sigma(a + b\sigma^2 + c\sigma^4)$ for both coefficient sets.

In [ ]:
sigma = np.linspace(0, 1.5, 500)

# Two coefficient sets
muon_coeffs = (3.4445, -4.7750, 2.0315)      # Muon: a+b+c ≈ 0.70
polar_coeffs = (15/8, -10/8, 3/8)             # Polar: a+b+c = 1.0

def quintic(s, a, b, c):
    return s * (a + b * s**2 + c * s**4)

p_muon = quintic(sigma, *muon_coeffs)
p_polar = quintic(sigma, *polar_coeffs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: the polynomials
ax1.plot(sigma, p_muon, 'b-', linewidth=2, label=f'Muon ({muon_coeffs[0]}, {muon_coeffs[1]}, {muon_coeffs[2]})')
ax1.plot(sigma, p_polar, 'r-', linewidth=2, label=f'Polar (15/8, -10/8, 3/8)')
ax1.plot(sigma, sigma, 'k--', alpha=0.5, label='y = σ (identity)')
ax1.axhline(y=1.0, color='gray', ls=':', alpha=0.5)
ax1.set_xlabel('σ (input singular value)')
ax1.set_ylabel('p(σ) (output singular value)')
ax1.set_title('Newton-Schulz Quintic Polynomial')
ax1.legend()
ax1.set_xlim(0, 1.5)
ax1.set_ylim(0, 1.5)

# Right: iterated convergence — apply p() repeatedly starting from σ₀
sigmas_to_track = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1]
n_iters = 10

for s0 in sigmas_to_track:
    trajectory = [s0]
    s = s0
    for _ in range(n_iters):
        s = quintic(s, *polar_coeffs)
        trajectory.append(s)
    ax2.plot(range(n_iters + 1), trajectory, 'o-', markersize=4, label=f'σ₀={s0}')

ax2.axhline(y=1.0, color='black', ls='--', alpha=0.5, label='target σ=1')
ax2.set_xlabel('NS Iteration')
ax2.set_ylabel('σ')
ax2.set_title('Polar Coefficients: All SVs → 1.0')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('quintic_polynomial.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 Newton-Schulz on a Real Matrix — Watching SVs Converge

Take a random 64×64 matrix and apply NS iteration step by step. Track all 64 singular values.

In [ ]:
run = wandb.init(
    project='mhc-nanogpt-blog',
    name='ns_convergence',
    tags=['newton-schulz', 'svd', 'blog-part1'],
    config={'matrix_size': 64, 'max_iters': 15}
)

torch.manual_seed(42)

def ns_step_by_step(G, coeffs, steps, normalize_fn):
    """Run NS iteration, recording SVs at each step."""
    a, b, c = coeffs
    X = G.float()
    X = normalize_fn(X)

    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T

    sv_history = []
    svs = torch.linalg.svdvals(X if not transposed else X.T)
    sv_history.append(svs.cpu().numpy())

    for i in range(steps):
        A = X @ X.T
        B = A @ X
        X = a * X + b * B + c * A @ B
        svs = torch.linalg.svdvals(X if not transposed else X.T)
        sv_history.append(svs.cpu().numpy())

    return sv_history

def frob_normalize(X, eps=1e-7):
    return X / (X.norm() + eps)

def spectral_normalize(X, eps=1e-7):
    return X / (torch.linalg.norm(X, ord=2) + eps)

G = torch.randn(64, 64)
max_iters = 15

sv_polar = ns_step_by_step(G.clone(), (15/8, -10/8, 3/8), max_iters, spectral_normalize)
sv_muon = ns_step_by_step(G.clone(), (3.4445, -4.7750, 2.0315), max_iters, frob_normalize)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Polar coefficients
sv_arr = np.array(sv_polar)
for j in range(sv_arr.shape[1]):
    axes[0].plot(range(max_iters + 1), sv_arr[:, j], alpha=0.4, linewidth=0.8)
axes[0].axhline(y=1.0, color='red', ls='--', linewidth=2, label='target σ=1')
axes[0].set_xlabel('NS Iteration')
axes[0].set_ylabel('Singular Value')
axes[0].set_title('Polar Coefficients (15/8, -10/8, 3/8)\nAll SVs → 1.0')
axes[0].legend()

# Muon coefficients
sv_arr = np.array(sv_muon)
for j in range(sv_arr.shape[1]):
    axes[1].plot(range(max_iters + 1), sv_arr[:, j], alpha=0.4, linewidth=0.8)
axes[1].axhline(y=0.6836, color='red', ls='--', linewidth=2, label='fixed point ≈ 0.68')
axes[1].set_xlabel('NS Iteration')
axes[1].set_ylabel('Singular Value')
axes[1].set_title('Muon Coefficients (3.4445, -4.7750, 2.0315)\nSVs equalized but ≠ 1')
axes[1].legend()

plt.tight_layout()
wandb.log({'ns_convergence': wandb.Image(fig)})
plt.savefig('ns_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

# Log per-iteration stats as line plots
for i in range(max_iters + 1):
    svs_p = sv_polar[i]
    svs_m = sv_muon[i]
    wandb.log({
        'ns_iter': i,
        'polar/sv_min': svs_p.min(),
        'polar/sv_max': svs_p.max(),
        'polar/sv_std': svs_p.std(),
        'polar/sv_mean': svs_p.mean(),
        'polar/condition': svs_p.max() / (svs_p.min() + 1e-8),
        'muon/sv_min': svs_m.min(),
        'muon/sv_max': svs_m.max(),
        'muon/sv_std': svs_m.std(),
        'muon/sv_mean': svs_m.mean(),
        'muon/condition': svs_m.max() / (svs_m.min() + 1e-8),
    })

print(f'Polar: final SV range [{sv_polar[-1].min():.4f}, {sv_polar[-1].max():.4f}]')
print(f'Muon:  final SV range [{sv_muon[-1].min():.4f}, {sv_muon[-1].max():.4f}]')

wandb.finish()

### 1.3 Why Does This Work? The Fixed-Point Analysis

For polar coefficients $(a, b, c) = (15/8, -10/8, 3/8)$:
- $p(1) = 1 \cdot (15/8 - 10/8 + 3/8) = 1.0$ ✓ (fixed point)
- $p'(1) = 0$ (super-attractive — quadratic convergence)

For Muon coefficients $(3.4445, -4.7750, 2.0315)$:
- $a + b + c \approx 0.701 \neq 1$ → $\sigma = 1$ is NOT a fixed point
- The actual fixed point is at $\sigma \approx 0.6836$
- But that's fine! Muon only needs the **direction**, not exact orthogonality

In [ ]:
# Verify the fixed points numerically
from scipy.optimize import brentq

def fixed_point_eq(s, a, b, c):
    return quintic(s, a, b, c) - s

# Polar: trivial fixed point at σ=1
fp_polar = brentq(fixed_point_eq, 0.5, 1.5, args=polar_coeffs)
print(f'Polar fixed point:  σ* = {fp_polar:.6f}')
print(f'  p(σ*) = {quintic(fp_polar, *polar_coeffs):.6f}')

# Muon: non-trivial fixed point
fp_muon = brentq(fixed_point_eq, 0.1, 0.99, args=muon_coeffs)
print(f'Muon fixed point:   σ* = {fp_muon:.6f}')
print(f'  p(σ*) = {quintic(fp_muon, *muon_coeffs):.6f}')

### 1.4 Convergence Speed: How Many Iterations Do You Need?

Compare the max SV deviation from the fixed point as a function of iteration count, for different matrix sizes and condition numbers.

In [ ]:
run = wandb.init(
    project='mhc-nanogpt-blog',
    name='ns_convergence_speed',
    tags=['newton-schulz', 'convergence', 'blog-part1'],
)

torch.manual_seed(0)
sizes = [4, 16, 64, 256]
max_steps = 20

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sz in sizes:
    G = torch.randn(sz, sz)
    sv_hist = ns_step_by_step(G, (15/8, -10/8, 3/8), max_steps, spectral_normalize)
    errors = [np.max(np.abs(svs - 1.0)) for svs in sv_hist]
    axes[0].semilogy(range(max_steps + 1), errors, 'o-', markersize=3, label=f'{sz}×{sz}')
    for i, err in enumerate(errors):
        wandb.log({f'convergence/polar_{sz}x{sz}_max_error': err, 'ns_step': i})

axes[0].set_xlabel('NS Iteration')
axes[0].set_ylabel('max |σᵢ - 1.0|')
axes[0].set_title('Polar: Convergence Speed vs Matrix Size')
axes[0].legend()
axes[0].axhline(y=1e-3, color='red', ls=':', label='1e-3 threshold')

# Same but for different condition numbers on 64x64
cond_numbers = [2, 10, 100, 1000]
for cond in cond_numbers:
    U, _, Vt = torch.linalg.svd(torch.randn(64, 64))
    svs_init = torch.linspace(1.0, cond, 64)
    G = U[:, :64] @ torch.diag(svs_init) @ Vt[:64, :]
    sv_hist = ns_step_by_step(G, (15/8, -10/8, 3/8), max_steps, spectral_normalize)
    errors = [np.max(np.abs(svs - 1.0)) for svs in sv_hist]
    axes[1].semilogy(range(max_steps + 1), errors, 'o-', markersize=3, label=f'κ={cond}')

axes[1].set_xlabel('NS Iteration')
axes[1].set_ylabel('max |σᵢ - 1.0|')
axes[1].set_title('Polar: Convergence Speed vs Condition Number (64×64)')
axes[1].legend()
axes[1].axhline(y=1e-3, color='red', ls=':', label='1e-3 threshold')

plt.tight_layout()
wandb.log({'convergence_speed': wandb.Image(fig)})
plt.savefig('ns_convergence_speed.png', dpi=150, bbox_inches='tight')
plt.show()

wandb.finish()

---
## Part 2: The Muon Optimizer — Momentum Orthogonalized by Newton-Schulz

### The idea

Standard optimizers (SGD, Adam) use the gradient direction as-is. The gradient's singular value spectrum can be wildly spread — some directions get huge updates, others tiny.

**Muon** orthogonalizes the momentum buffer via Newton-Schulz before using it as the update. This equalizes the update's singular values, so the optimizer makes equal-magnitude steps in all directions of the weight space.

It's like applying a preconditioning that says: *"I don't care how big the gradient is in each direction — just tell me the direction."*

### 2.1 Visualize: What Muon Does to a Single Gradient

Take a gradient matrix from a real model layer and show its SV spectrum before vs after Muon's NS processing.

In [ ]:
run = wandb.init(
    project='mhc-nanogpt-blog',
    name='muon_gradient_spectrum',
    tags=['muon', 'gradient-spectrum', 'blog-part1'],
)

torch.manual_seed(42)

# Create a small model and compute a real gradient
config = TrainConfig(
    n_layers=4, d_model=256, n_heads=4, d_ff=1024,
    vocab_size=256, context_len=64,
    dropout=0.0, bias=False, use_mhc=False, compile=False,
)
model = GPT(config).to(device)

# Forward + backward to get gradients
x = torch.randint(0, 256, (4, 64), device=device)
y = torch.randint(0, 256, (4, 64), device=device)
_, loss = model(x, y)
loss.backward()

# Collect gradients from weight matrices
grad_layers = []
for name, p in model.named_parameters():
    if p.grad is not None and p.ndim == 2 and 'c_attn' in name:
        grad_layers.append((name, p.grad.clone()))

# Show before/after for first few layers
n_show = min(4, len(grad_layers))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))

for i, (name, grad) in enumerate(grad_layers[:n_show]):
    # Before: raw gradient SVs
    svs_before = torch.linalg.svdvals(grad.float()).cpu().numpy()

    # After: Muon-processed (NS orthogonalization)
    grad_ns = newton_schulz_muon(grad, steps=5)
    svs_after = torch.linalg.svdvals(grad_ns.float()).cpu().numpy()

    axes[0, i].bar(range(len(svs_before)), svs_before, color='steelblue', alpha=0.8)
    axes[0, i].set_title(f'Raw gradient\n{name.split(".")[-2]}.{name.split(".")[-1]}', fontsize=10)
    axes[0, i].set_xlabel('SV index')
    if i == 0:
        axes[0, i].set_ylabel('Singular Value')

    axes[1, i].bar(range(len(svs_after)), svs_after, color='coral', alpha=0.8)
    axes[1, i].set_title(f'After NS (Muon)', fontsize=10)
    axes[1, i].set_xlabel('SV index')
    if i == 0:
        axes[1, i].set_ylabel('Singular Value')

    # Log stats
    wandb.log({
        f'gradient_sv/{name}/before_condition': svs_before.max() / (svs_before.min() + 1e-8),
        f'gradient_sv/{name}/after_condition': svs_after.max() / (svs_after.min() + 1e-8),
        f'gradient_sv/{name}/before_std': svs_before.std(),
        f'gradient_sv/{name}/after_std': svs_after.std(),
    })

plt.suptitle('Gradient SV Spectrum: Raw vs Muon-Processed', fontsize=14, y=1.02)
plt.tight_layout()
wandb.log({'gradient_spectrum': wandb.Image(fig)})
plt.savefig('muon_gradient_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

model.zero_grad()
wandb.finish()

### 2.2 Training Comparison: AdamW vs Muon

Train a small transformer (4 layers, 256 hidden) on random token sequences for 500 steps. Compare:
- **Loss curves**
- **Gradient update spectrum** (condition number of the update direction over training)

This isn't about reaching good loss on real data — it's about showing the *optimizer dynamics*.

In [ ]:
def make_demo_config(optimizer_name):
    return TrainConfig(
        n_layers=4, d_model=256, n_heads=4, d_ff=1024,
        vocab_size=2048, context_len=128,
        dropout=0.0, bias=False, use_mhc=False, compile=False,
        optimizer=optimizer_name,
        lr=6e-4, min_lr=6e-5,
        muon_lr=0.02, muon_min_lr=0.002, muon_momentum=0.95, muon_ns_steps=5,
        weight_decay=0.1,
        batch_size=16, grad_accum_steps=1,
        max_steps=500, warmup_steps=50,
    )


def train_demo(config, run_name):
    """Minimal training loop for the demo — synthetic data, no eval."""
    import math

    torch.manual_seed(42)

    run = wandb.init(
        project='mhc-nanogpt-blog',
        name=run_name,
        tags=['training-comparison', 'blog-part1', config.optimizer],
        config={
            'optimizer': config.optimizer,
            'n_layers': config.n_layers,
            'd_model': config.d_model,
            'lr': config.lr,
            'muon_lr': config.muon_lr,
            'batch_size': config.batch_size,
            'max_steps': config.max_steps,
        },
    )

    model = GPT(config).to(device)
    print(f'{run_name}: {model.count_parameters():,} params')

    # Build optimizer
    if config.optimizer == 'adamw':
        decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.ndim >= 2]
        nodecay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.ndim < 2]
        optimizer = torch.optim.AdamW([
            {'params': decay_params, 'weight_decay': config.weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ], lr=config.lr, betas=(config.beta1, config.beta2))
        muon_opt = None
    else:
        muon_params, adamw_decay, adamw_nodecay = model.get_param_groups()
        muon_opt = Muon(
            [{'params': muon_params}],
            lr=config.muon_lr, momentum=config.muon_momentum,
            ns_steps=config.muon_ns_steps,
        )
        optimizer = torch.optim.AdamW([
            {'params': adamw_decay, 'weight_decay': config.weight_decay},
            {'params': adamw_nodecay, 'weight_decay': 0.0},
        ], lr=config.lr, betas=(config.beta1, config.beta2))

    losses = []
    update_conditions = []

    for step in range(config.max_steps):
        # LR schedule
        if step < config.warmup_steps:
            lr = config.lr * (step + 1) / config.warmup_steps
        else:
            progress = (step - config.warmup_steps) / (config.max_steps - config.warmup_steps)
            lr = config.min_lr + 0.5 * (config.lr - config.min_lr) * (1 + math.cos(math.pi * progress))
        for g in optimizer.param_groups:
            g['lr'] = lr
        if muon_opt is not None:
            muon_lr = config.muon_min_lr + 0.5 * (config.muon_lr - config.muon_min_lr) * (
                1 + math.cos(math.pi * max(0, (step - config.warmup_steps)) / (config.max_steps - config.warmup_steps))
            ) if step >= config.warmup_steps else config.muon_lr * (step + 1) / config.warmup_steps
            for g in muon_opt.param_groups:
                g['lr'] = muon_lr

        # Synthetic data: random tokens
        x = torch.randint(0, config.vocab_size, (config.batch_size, config.context_len), device=device)
        y = torch.randint(0, config.vocab_size, (config.batch_size, config.context_len), device=device)

        model.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        # Snapshot gradient spectrum before optimizer step
        if step % 50 == 0:
            for name, p in model.named_parameters():
                if p.grad is not None and p.ndim == 2 and 'c_attn.weight' in name:
                    svs = torch.linalg.svdvals(p.grad.float())
                    cond = (svs.max() / (svs.min() + 1e-8)).item()
                    update_conditions.append(cond)
                    wandb.log({
                        'update_spectrum/grad_condition': cond,
                        'update_spectrum/grad_sv_std': svs.std().item(),
                    }, step=step)
                    break

        if muon_opt is not None:
            muon_opt.step()
            # Log Muon's post-NS spectrum
            if step % 50 == 0:
                stats = muon_opt.get_spectrum_stats()
                if stats:
                    s = list(stats.values())[0]
                    wandb.log({
                        'update_spectrum/muon_condition': s['max'] / (s['min'] + 1e-8),
                        'update_spectrum/muon_sv_std': s['std'],
                    }, step=step)

        optimizer.step()

        losses.append(loss.item())
        if step % 10 == 0:
            wandb.log({'train/loss': loss.item(), 'train/lr': lr}, step=step)
        if step % 100 == 0:
            print(f'  step {step:>4d} | loss {loss.item():.4f} | lr {lr:.2e}')

    wandb.finish()
    return losses

print('Config ready. Training will start in the next cells.')

In [ ]:
# Train with AdamW
print('=== Training with AdamW ===')
adamw_config = make_demo_config('adamw')
adamw_losses = train_demo(adamw_config, 'demo_adamw')

In [ ]:
# Train with Muon
print('=== Training with Muon ===')
muon_config = make_demo_config('muon')
muon_losses = train_demo(muon_config, 'demo_muon')

In [ ]:
# Plot comparison
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

window = 20
adamw_smooth = np.convolve(adamw_losses, np.ones(window)/window, mode='valid')
muon_smooth = np.convolve(muon_losses, np.ones(window)/window, mode='valid')

ax.plot(adamw_smooth, color='#1f77b4', linewidth=1.5, label='AdamW', alpha=0.9)
ax.plot(muon_smooth, color='#ff7f0e', linewidth=1.5, label='Muon', alpha=0.9)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (smoothed)')
ax.set_title('Training Loss: AdamW vs Muon (4-layer, 256-dim transformer)')
ax.legend()

plt.tight_layout()
plt.savefig('adamw_vs_muon_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'AdamW final loss: {np.mean(adamw_losses[-50:]):.4f}')
print(f'Muon  final loss: {np.mean(muon_losses[-50:]):.4f}')

### 2.3 The Key Insight: Update Spectrum

The W&B plots above show `update_spectrum/grad_condition` (condition number of the raw gradient) vs `update_spectrum/muon_condition` (condition number after NS processing).

The raw gradient's condition number can be 100–10,000×. After Muon's NS processing, it drops to ~1.0 — all singular values are equalized.

This is the Muon trick: **use the SVD-free orthogonalization to precondition the optimizer**.

---
## Summary

1. **Newton-Schulz iteration** replaces explicit SVD with a cheap matrix polynomial. The right coefficients make $\sigma = 1$ a stable fixed point, giving you the polar decomposition in 5–10 iterations of purely matmul ops.

2. **Muon** applies this to the momentum buffer, equalizing update singular values. The optimizer doesn't care about gradient magnitude — only direction.

3. Both ideas compose: DeepSeek V4 uses Newton-Schulz for Muon-style optimization *and* for manifold retraction on routing matrices. Part 2 of this blog will cover the manifold constraint.

All plots are on W&B under project `mhc-nanogpt-blog`. Check the dashboard for interactive versions!